# TensorFlow 花卉识别模型训练

本 Notebook 演示如何使用 TensorFlow/Keras 训练花卉识别模型，并将其转换为 TensorFlow Lite 格式，以便在 Android 应用中使用。

## 实验目标

1. 了解机器学习基础
2. 了解 TensorFlow 和 TensorFlow Lite
3. 使用 TensorFlow/Keras 训练花卉分类模型
4. 将模型转换为 .tflite 格式
5. 在 Android 应用中验证生成的模型

## 步骤 1: 环境准备和导入依赖库

In [ ]:
# 导入必要的库
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import os

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

## 步骤 2: 准备数据集

我们使用经典的花卉数据集（包含5个类别：daisy, dandelion, roses, sunflowers, tulips）。

In [ ]:
# 下载花卉数据集
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
data_dir = tf.keras.utils.get_file('flower_photos', origin=dataset_url, untar=True)
data_dir = pathlib.Path(data_dir)

print(f"数据集路径: {data_dir}")

# 统计图片数量
image_count = len(list(data_dir.glob('*/*.jpg')))
print(f"总图片数: {image_count}")

# 查看类别
class_names = sorted([item.name for item in data_dir.glob('*') if item.is_dir()])
print(f"\n类别列表: {class_names}")
print(f"类别数: {len(class_names)}")

# 统计每个类别的图片数
for name in class_names:
    count = len(list(data_dir.glob(f'{name}/*.jpg')))
    print(f"  {name}: {count} 张图片")

In [ ]:
# 显示几张示例图片
import PIL
import PIL.Image

# 查看玫瑰的示例图片
roses = list(data_dir.glob('roses/*'))
print("示例玫瑰图片:")
PIL.Image.open(str(roses[0]))

## 步骤 3: 创建数据加载管道 (Data Pipeline)

In [ ]:
# 参数设置
batch_size = 32
img_height = 224
img_width = 224

# 从目录创建训练数据集 (80% 训练)
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

# 从目录创建验证数据集 (20% 验证)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

print(f"\n训练批次数: {tf.data.experimental.cardinality(train_ds).numpy()}")
print(f"验证批次数: {tf.data.experimental.cardinality(val_ds).numpy()}")

In [ ]:
# 显示训练集中的一些图片和标签
class_names = train_ds.class_names
print(f"类别名称: {class_names}")

plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")
plt.show()

## 步骤 4: 构建模型 (迁移学习 - 使用 MobileNetV2)

为了在移动设备上高效运行，我们使用轻量级的 MobileNetV2 作为基础模型进行迁移学习。

In [ ]:
# 数据增强层（有助于防止过拟合）
data_augmentation = keras.Sequential(
  [
    layers.experimental.preprocessing.RandomFlip("horizontal", input_shape=(img_height, img_width, 3)),
    layers.experimental.preprocessing.RandomRotation(0.1),
    layers.experimental.preprocessing.RandomZoom(0.1),
  ]
)

# 可视化数据增强效果
plt.figure(figsize=(10, 10))
for images, _ in train_ds.take(1):
    for i in range(9):
        augmented_images = data_augmentation(images)
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(augmented_images[0].numpy().astype("uint8"))
        plt.axis("off")
plt.show()

In [ ]:
# 构建迁移学习模型
num_classes = len(class_names)

# 加载预训练的 MobileNetV2 作为基础模型
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(img_height, img_width, 3),
    include_top=False,
    weights='imagenet'
)

# 冻结基础模型（不更新预训练权重）
base_model.trainable = False

# 构建完整模型
inputs = keras.Input(shape=(img_height, img_width, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = keras.Model(inputs, outputs)

# 显示模型结构
model.summary()

## 步骤 5: 编译模型

In [ ]:
# 编译模型
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

print("模型编译完成！")

## 步骤 6: 训练模型

In [ ]:
# 配置性能优化（预取数据）
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# 训练模型
epochs = 10
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs
)

## 步骤 7: 评估模型并可视化训练结果

In [ ]:
# 可视化训练和验证准确率/损失
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(epochs)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

# 打印最终准确率
print(f"\n最终训练准确率: {acc[-1]:.4f}")
print(f"最终验证准确率: {val_acc[-1]:.4f}")

In [ ]:
# 使用混淆矩阵进行更详细的评估
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

y_true = []
y_pred = []

for images, labels in val_ds:
    predictions = model.predict(images)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

# 混淆矩阵
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

# 详细分类报告
print("\n分类报告:")
print(classification_report(y_true, y_pred, target_names=class_names))

## 步骤 8: 保存 Keras 模型

In [ ]:
# 保存完整的 Keras 模型（用于继续训练或推理）
model.save('flower_model.h5')
print("Keras 模型已保存为 flower_model.h5")

# 也保存为 SavedModel 格式（推荐用于 TensorFlow Lite 转换）
tf.saved_model.save(model, 'flower_saved_model')
print("SavedModel 已保存到 flower_saved_model/ 目录")

## 步骤 9: 转换为 TensorFlow Lite 格式

现在我们将训练好的模型转换为 TensorFlow Lite 格式，使其能在移动设备上高效运行。

In [ ]:
# 创建 TensorFlow Lite 转换器
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# 应用优化（可选 - 量化模型以减小大小）
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# 转换模型
tflite_model = converter.convert()

# 保存 .tflite 文件
tflite_model_path = 'FlowerModel.tflite'
with open(tflite_model_path, 'wb') as f:
    f.write(tflite_model)

print(f"TensorFlow Lite 模型已保存: {tflite_model_path}")
print(f"模型大小: {os.path.getsize(tflite_model_path) / 1024:.2f} KB")
print(f"模型大小: {os.path.getsize(tflite_model_path) / 1024 / 1024:.4f} MB")

## 步骤 10: 验证 TensorFlow Lite 模型

确保转换后的模型能正确推理。

In [ ]:
# 加载并验证 TensorFlow Lite 模型
interpreter = tf.lite.Interpreter(model_path='FlowerModel.tflite')
interpreter.allocate_tensors()

# 获取输入输出详情
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("=== 模型详情 ===")
print(f"\n输入形状: {input_details[0]['shape']}")
print(f"输入类型: {input_details[0]['dtype']}")
print(f"\n输出形状: {output_details[0]['shape']}")
print(f"输出类型: {output_details[0]['dtype']}")
print(f"\n类别数: {output_details[0]['shape'][1]}")
print(f"类别名称: {class_names}")

In [ ]:
# 用一张图片测试 TensorFlow Lite 模型
from tensorflow.keras.preprocessing import image

# 获取测试图片
test_image_path = str(roses[0])
img = image.load_img(test_image_path, target_size=(img_height, img_width))

# 预处理图片
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = tf.keras.applications.mobilenet_v2.preprocess_input(img_array)

# 设置输入张量并推理
interpreter.set_tensor(input_details[0]['index'], img_array.astype(input_details[0]['dtype']))
interpreter.invoke()

# 获取输出
output_data = interpreter.get_tensor(output_details[0]['index'])
predictions = output_data[0]

# 显示结果
print("=== TensorFlow Lite 模型推理结果 ===")
print(f"\n测试图片: {test_image_path}")
print(f"\n预测概率分布:")
for i, class_name in enumerate(class_names):
    print(f"  {class_name}: {predictions[i]:.4f} ({predictions[i]*100:.2f}%)")

predicted_class = np.argmax(predictions)
print(f"\n预测类别: {class_names[predicted_class]}")
print(f"置信度: {predictions[predicted_class]:.4f} ({predictions[predicted_class]*100:.2f}%)")

# 显示图片
plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.title(f"Predicted: {class_names[predicted_class]} ({predictions[predicted_class]*100:.1f}%)")
plt.axis('off')
plt.show()

## 步骤 11: 为模型添加元数据（可选）

使用 TensorFlow Lite Metadata Writer 为模型添加元数据，这样 Android Studio ML Model Binding 可以自动生成接口。

In [ ]:
try:
    from tflite_support.metadata_writers import image_classifier
    from tflite_support.metadata_writers import writer_utils
    
    # 创建标签文件
    with open('labels.txt', 'w') as f:
        for name in class_names:
            f.write(name + '\n')
    print(f"标签文件已保存: labels.txt")
    print(f"标签内容: {class_names}")
    
except ImportError:
    print("tflite_support 未安装，跳过元数据写入。")
    print("你可以运行: pip install tflite-support")
    
    # 仍然创建简单的标签文件
    with open('labels.txt', 'w') as f:
        for name in class_names:
            f.write(name + '\n')
    print(f"\n标签文件已保存: labels.txt")
    print(f"标签内容: {class_names}")

## 实验总结

恭喜！您已成功完成以下步骤：

### 1. 机器学习基础
- 了解了监督学习中图像分类的基本概念
- 掌握了使用 `tf.keras.preprocessing.image_dataset_from_directory` 创建数据加载器
- 学习了数据增强技术来防止过拟合

### 2. TensorFlow / Keras 应用
- 使用 TensorFlow 2.x 和 Keras API 构建模型
- 使用 MobileNetV2 预训练模型进行迁移学习
- 冻结基础模型并添加自定义分类层

### 3. 模型训练与评估
- 训练准确率: ~95%+
- 验证准确率: ~90%+
- 使用混淆矩阵和分类报告评估模型

### 4. TensorFlow Lite 转换
- 使用 `tf.lite.TFLiteConverter` 将 Keras 模型转换为 .tflite 格式
- 应用模型量化优化（`Optimize.DEFAULT`）
- 验证转换后的模型推理正确性

### 5. 下一步
- 将生成的 `FlowerModel.tflite` 文件复制到 Android 项目的 `app/src/main/ml/` 目录
- 在 Android Studio 中使用 ML Model Binding 自动生成模型接口
- 在真机上运行并测试花卉识别应用

---

### 生成的文件列表
- `flower_model.h5` - Keras 模型文件 (H5 格式)
- `flower_saved_model/` - SavedModel 格式目录
- `FlowerModel.tflite` - TensorFlow Lite 模型 (**用于 Android 项目**)
- `labels.txt` - 类别标签文件